# Big NSS: Predicting Truck Derates

Clean the Big NSS trucking dataset and build models to predict engine derates using J1939 sensor data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              accuracy_score, precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

EQUIP_COL    = 'EquipmentID'
OIL_COLS     = ['EngineOilTemperature', 'EngineOilPressure']
LTD_COLS     = ['DistanceLtd', 'FuelLtd', 'EngineTimeLtd']
FEATURE_COLS = LTD_COLS + OIL_COLS
TARGET_COL   = 'Derate'

df = pd.read_csv('big_nss.csv', index_col=0)
df['EventTimeStamp'] = pd.to_datetime(df['EventTimeStamp'])
if df[TARGET_COL].dtype == object:
    df[TARGET_COL] = df[TARGET_COL].map({'True': True, 'False': False})
df[TARGET_COL] = df[TARGET_COL].astype(bool)
df = df.sort_values([EQUIP_COL, 'EventTimeStamp']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Derate counts: {df[TARGET_COL].value_counts().to_dict()}')
df.head()

## 1. Explore Data & Visualize

Look at null counts, then compare the 5 measurement columns between trucks in derate vs not.

In [ ]:
print('Null counts per column:')
print(df.isna().sum().to_string())

In [ ]:
derate_yes = df[df[TARGET_COL]][FEATURE_COLS]
derate_no  = df[~df[TARGET_COL]][FEATURE_COLS]

fig, axes = plt.subplots(1, 5, figsize=(24, 7))
fig.suptitle('Derate vs No Derate -- 5 Measurement Columns', fontsize=15, fontweight='bold', y=1.02)
COLORS = ['#4C72B0', '#DD4444']

for ax, col in zip(axes, FEATURE_COLS):
    no_vals  = derate_no[col].dropna()
    yes_vals = derate_yes[col].dropna()
    bp = ax.boxplot([no_vals, yes_vals], patch_artist=True, widths=0.55,
                    medianprops=dict(color='white', linewidth=2.5),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['No Derate', 'Derate'], fontsize=11)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    if len(no_vals) > 0 and len(yes_vals) > 0:
        stat, p = stats.mannwhitneyu(no_vals, yes_vals, alternative='two-sided')
        stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.set_xlabel(f'p={p:.4f} {stars}', fontsize=9, color='dimgray')

plt.tight_layout()
plt.savefig('derate_comparison_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Clean Data

1. Drop rows with no useful measurements at all
2. Drop trucks missing 2+ Ltd columns entirely (for the whole equipment ID)
3. Impute partially-missing Ltd values using the other Ltd columns (IterativeImputer)
4. Fill remaining Ltd gaps by linear interpolation within each truck
5. Impute missing EngineOil values using Ltd + available EngineOil data

In [ ]:
oil_all_null = df[OIL_COLS].isna().all(axis=1)
ltd_all_null = df[LTD_COLS].isna().all(axis=1)
rows_before = len(df)
df = df[~(oil_all_null & ltd_all_null)].reset_index(drop=True)
print(f'Dropped: {rows_before - len(df):,} rows | Remaining: {len(df):,} rows')

In [ ]:
all_nan_per_equip = (
    df.groupby(EQUIP_COL)[LTD_COLS]
    .agg(lambda x: x.isna().all())
    .sum(axis=1)
)
equip_to_drop = all_nan_per_equip[all_nan_per_equip >= 2].index.tolist()
rows_before  = len(df)
equip_before = df[EQUIP_COL].nunique()
df = df[~df[EQUIP_COL].isin(equip_to_drop)].reset_index(drop=True)
print(f'Equipment IDs dropped  : {len(equip_to_drop):,}')
print(f'Equipment IDs remaining: {df[EQUIP_COL].nunique():,}  (was {equip_before:,})')
print(f'Rows dropped           : {rows_before - len(df):,}')
print(f'Rows remaining         : {len(df):,}')

In [ ]:
complete_mask = df[LTD_COLS].notna().all(axis=1)
partial_mask  = df[LTD_COLS].isna().any(axis=1) & ~df[LTD_COLS].isna().all(axis=1)
print(f'Complete rows (all 3 Ltd): {complete_mask.sum():,}')
print(f'Partial rows  (1-2 Ltd)  : {partial_mask.sum():,}')
imputer_ltd = IterativeImputer(estimator=LinearRegression(), max_iter=10, random_state=42, min_value=0)
imputer_ltd.fit(df.loc[complete_mask, LTD_COLS])
if partial_mask.sum() == 0:
    print('No partial Ltd rows to impute.')
else:
    nulls_before = df.loc[partial_mask, LTD_COLS].isna().sum()
    df.loc[partial_mask, LTD_COLS] = imputer_ltd.transform(df.loc[partial_mask, LTD_COLS])
    nulls_after = df.loc[partial_mask, LTD_COLS].isna().sum()
    for col in LTD_COLS:
        print(f'  {col:20s}: {nulls_before[col]:,} filled -> {nulls_after[col]:,} remaining')

In [ ]:
nulls_before = df[LTD_COLS].isna().sum()
for col in LTD_COLS:
    df[col] = df.groupby(EQUIP_COL)[col].transform(lambda s: s.interpolate(method='linear'))
nulls_after = df[LTD_COLS].isna().sum()
for col in LTD_COLS:
    filled = nulls_before[col] - nulls_after[col]
    print(f'  {col:20s}: {filled:,} filled | {nulls_after[col]:,} remaining')
print(f'Total Ltd nulls remaining: {df[LTD_COLS].isna().sum().sum():,}')

In [ ]:
ALL_COLS = LTD_COLS + OIL_COLS
both_oil_null = df[OIL_COLS].isna().all(axis=1)
one_oil_null  = df[OIL_COLS].isna().any(axis=1) & ~both_oil_null
print(f'Both null (predict from Ltd only)        : {both_oil_null.sum():,} rows')
print(f'One null  (predict from Ltd + other oil) : {one_oil_null.sum():,} rows')
print(f'Both present (no action needed)          : {(~df[OIL_COLS].isna().any(axis=1)).sum():,} rows')
complete_mask = df[ALL_COLS].notna().all(axis=1)
imputer_oil = IterativeImputer(estimator=LinearRegression(), max_iter=10, random_state=42)
imputer_oil.fit(df.loc[complete_mask, ALL_COLS])
needs_impute = df[OIL_COLS].isna().any(axis=1)
if needs_impute.sum() == 0:
    print('No EngineOil nulls remaining -- nothing to impute.')
else:
    nulls_before = df.loc[needs_impute, OIL_COLS].isna().sum()
    imputed = pd.DataFrame(
        imputer_oil.transform(df.loc[needs_impute, ALL_COLS]),
        columns=ALL_COLS, index=df.loc[needs_impute].index
    )
    df.loc[needs_impute, OIL_COLS] = imputed[OIL_COLS]
    nulls_after = df.loc[needs_impute, OIL_COLS].isna().sum()
    for col in OIL_COLS:
        filled = nulls_before[col] - nulls_after[col]
        print(f'  {col:25s}: {filled:,} filled | {nulls_after[col]:,} remaining')
print(f'Total EngineOil nulls remaining: {df[OIL_COLS].isna().sum().sum():,}')

## 3. Current Derate Prediction (Classification)

**Target:** `Derate` (True/False) -- is the truck in a full derate right now?

**Features:** The 3 Ltd columns (odometer, fuel, engine hours) and 2 EngineOil columns.

**Train/test split:** 80% train, 20% held-back test. Scoring on the unseen test set gives an honest measure of real-world performance.

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL].astype(int)
clean_mask = X.notna().all(axis=1) & y.notna()
X_clean = X[clean_mask]
y_clean = y[clean_mask]
print(f'Rows before null drop: {len(X):,}')
print(f'Rows after null drop : {len(X_clean):,}  ({(~clean_mask).sum():,} dropped)')
print(f'Derate rate          : {y_clean.mean()*100:.1f}%')
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, stratify=y_clean, random_state=42
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Training rows: {len(X_train):,} | Test rows: {len(X_test):,}')
print(f'Derate rate -- train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%')

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)
lr_probs = lr.predict_proba(X_test_sc)[:, 1]

print('-- Logistic Regression ------------------------------------------')
print(classification_report(y_test, lr_preds, target_names=['No Derate', 'Derate']))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}')

coefficients = lr.coef_[0]
conf_int     = 1.96 * np.sqrt(np.diag(np.linalg.inv(X_train_sc.T @ X_train_sc)))
odds_ratios = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Odds Ratio': np.exp(coefficients),
    'CI Lower'  : np.exp(coefficients - conf_int),
    'CI Upper'  : np.exp(coefficients + conf_int),
    'Direction' : ['Increases derate risk' if c > 0 else 'Decreases derate risk'
                   for c in coefficients]
}).sort_values('Odds Ratio', ascending=False).round(3)
display(odds_ratios)

fig, ax = plt.subplots(figsize=(8, 4))
colors_or = ['#DD4444' if or_ > 1 else '#4C72B0' for or_ in odds_ratios['Odds Ratio']]
ax.barh(odds_ratios['Feature'], odds_ratios['Odds Ratio'], color=colors_or, alpha=0.8)
ax.errorbar(odds_ratios['Odds Ratio'], odds_ratios['Feature'],
            xerr=[odds_ratios['Odds Ratio'] - odds_ratios['CI Lower'],
                  odds_ratios['CI Upper']   - odds_ratios['Odds Ratio']],
            fmt='none', color='black', capsize=4)
ax.axvline(x=1, color='black', linestyle='--', lw=1.5)
ax.set_xlabel('Odds Ratio  (>1 = increases derate risk, <1 = decreases it)')
ax.set_title('Logistic Regression Odds Ratios with 95% CI')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

print('-- Random Forest ------------------------------------------------')
print(classification_report(y_test, rf_preds, target_names=['No Derate', 'Derate']))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, label in zip(axes[:2], [lr_preds, rf_preds],
                             ['Logistic Regression', 'Random Forest']):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, ax=ax,
        display_labels=['No Derate', 'Derate'], colorbar=False)
    ax.set_title(label)

for probs, label, color in [(lr_probs, 'Logistic Regression', '#4C72B0'),
                             (rf_probs, 'Random Forest', '#DD4444')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[2].plot(fpr, tpr, label=f'{label}  (AUC={auc:.3f})', color=color, lw=2)
axes[2].plot([0, 1], [0, 1], 'k--', lw=1, label='Random guess')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curves')
axes[2].legend()
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()
importances.plot.barh(ax=ax, color='steelblue')
ax.set_title('Random Forest -- Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
results = {}
for name, preds, probs in [('Logistic Regression', lr_preds, lr_probs),
                            ('Random Forest',       rf_preds, rf_probs)]:
    results[name] = {
        'Accuracy' : accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall'   : recall_score(y_test, preds, zero_division=0),
        'F1 Score' : f1_score(y_test, preds, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_test, probs)
    }
summary = pd.DataFrame(results).T.round(3)
display(summary)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(summary.columns))
width = 0.35
for i, (model, row) in enumerate(summary.iterrows()):
    bars = ax.bar(x + i*width, row.values, width, label=model,
                  color=['#4C72B0', '#DD4444'][i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x + width/2)
ax.set_xticklabels(summary.columns)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Accuracy Comparison')
ax.legend()
ax.axhline(0.5, color='gray', linestyle='--', lw=1, alpha=0.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

for model, row in summary.iterrows():
    print(f'\n{model}')
    print(f'  Accuracy  {row["Accuracy"]:.1%} -- of all test rows the model got this fraction right')
    print(f'  Precision {row["Precision"]:.1%} -- when it flagged DERATE it was correct this often')
    print(f'  Recall    {row["Recall"]:.1%} -- of real derates it caught this fraction')
    print(f'  F1        {row["F1 Score"]:.1%} -- balanced score combining precision and recall')
    print(f'  ROC-AUC   {row["ROC-AUC"]:.3f} -- 0.5=random | 0.7=ok | 0.8=good | 0.9+=excellent')

## 4. Future Derate Prediction (Early Warning)

Instead of predicting if a truck is in derate *right now*, predict if it *will* enter derate within the next N hours. This is the operationally useful question -- you want to intervene *before* the derate, not after.

In [ ]:
def create_lookahead_target(df, hours):
    derate_times = (
        df[df[TARGET_COL]][['EquipmentID', 'EventTimeStamp']]
        .assign(derate_time=lambda x: pd.to_datetime(x['EventTimeStamp']))
        .drop(columns='EventTimeStamp')
        .sort_values(['EquipmentID', 'derate_time'])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    df_sorted = (
        df[['EquipmentID', 'EventTimeStamp']]
        .assign(EventTimeStamp=lambda x: pd.to_datetime(x['EventTimeStamp']))
        .sort_values(['EquipmentID', 'EventTimeStamp'])
    )
    orig_idx  = df_sorted.index
    df_sorted = df_sorted.reset_index(drop=True)
    merged = pd.merge_asof(
        df_sorted, derate_times,
        left_on='EventTimeStamp', right_on='derate_time',
        by='EquipmentID', direction='forward'
    )
    delta_hours = (merged['derate_time'] - merged['EventTimeStamp']).dt.total_seconds() / 3600
    target = pd.Series(
        ((delta_hours > 0) & (delta_hours <= hours)).values,
        index=orig_idx
    )
    return target.reindex(df.index).fillna(False)

print('create_lookahead_target defined.')

In [ ]:
WINDOWS = [
    ('12 hours',  12),
    ('1 day',     24),
    ('3 days',    72),
    ('7 days',   168),
    ('14 days',  336),
    ('21 days',  504),
]
auc_results = {}

for label, hours in WINDOWS:
    y_look       = create_lookahead_target(df, hours).astype(int)
    not_derating = ~df[TARGET_COL].values
    X_w = df.loc[not_derating, FEATURE_COLS]
    y_w = y_look[not_derating]
    clean = X_w.notna().all(axis=1)
    X_w = X_w[clean]
    y_w = y_w[clean]
    pos_rate = y_w.mean() * 100
    print(f'{label:10s} | positives: {y_w.sum():,} ({pos_rate:.1f}%) | rows: {len(y_w):,}')
    if y_w.sum() < 10:
        print(f'           Skipping -- too few positive examples.')
        continue
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_w, y_w, test_size=0.2, stratify=y_w, random_state=42
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    auc_results[label] = {'auc': auc, 'pos_rate': pos_rate}
    print(f'           ROC-AUC = {auc:.4f}')

In [ ]:
labels = list(auc_results.keys())
aucs   = [v['auc'] for v in auc_results.values()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(labels, aucs, marker='o', color='steelblue', lw=2.5, markersize=10, zorder=3)
for x, (label, auc) in enumerate(zip(labels, aucs)):
    ax.annotate(f'{auc:.3f}', (x, auc), textcoords='offset points',
                xytext=(0, 12), ha='center', fontweight='bold', fontsize=11)
ax.axhline(0.5, color='gray',    linestyle='--', lw=1.5, label='Random guess (0.5)')
ax.axhline(0.7, color='#f39c12', linestyle='--', lw=1.0, alpha=0.8, label='Fair (0.7)')
ax.axhline(0.8, color='#2ecc71', linestyle='--', lw=1.0, alpha=0.8, label='Good (0.8)')
ax.fill_between(range(len(labels)), 0.5, aucs, alpha=0.1, color='steelblue')
ax.set_ylim(0.4, 1.05)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_xlabel('How Far Ahead We Are Predicting', fontsize=12)
ax.set_ylabel('ROC-AUC Score', fontsize=12)
ax.set_title('Early Warning Signal: How Far Ahead Can We Predict a Derate?', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
key_1day = next((k for k in auc_results if '1 day' in k.lower()), None)

if key_1day is None:
    print('No 1-day result found. Keys:', list(auc_results.keys()))
else:
    auc_1day = auc_results[key_1day]['auc']
    pos_rate = auc_results[key_1day]['pos_rate']

    print('=' * 65)
    print(f'1-DAY LOOKAHEAD MODEL  --  AUC = {auc_1day:.3f}')
    print('=' * 65)
    print('WHAT THIS MEANS:')
    print(f'  If you randomly pick one truck about to derate (within 24 h)')
    print(f'  and one that is NOT, the model correctly ranks the about-to-derate')
    print(f'  truck as higher risk {auc_1day*100:.1f}% of the time.')
    print(f'  Only {pos_rate:.1f}% of non-derating rows fall within 24 hours of a derate.')
    print()

    if auc_1day >= 0.90:
        verdict = 'EXCELLENT -- strong enough to act on operationally.'
    elif auc_1day >= 0.80:
        verdict = 'GOOD -- reliable enough to flag high-risk trucks for inspection.'
    elif auc_1day >= 0.70:
        verdict = 'FAIR -- some signal present but expect meaningful false alarms.'
    elif auc_1day >= 0.60:
        verdict = 'WEAK -- marginal; consider adding more sensor features.'
    else:
        verdict = 'POOR -- these features alone cannot reliably predict 24 hours out.'
    print(f'VERDICT: {verdict}')
    print()
    print('WHAT ROC-AUC DOES NOT TELL YOU:')
    print('  - How many false alarms you will get (depends on threshold chosen)')
    print('  - Which trucks to inspect today (need to run on live data)')
    print('  - The cost of missing a derate vs a false alarm inspection')
    print()

    y_look_1d    = create_lookahead_target(df, 24).astype(int)
    not_derating = ~df[TARGET_COL].values
    clean        = df.loc[not_derating, FEATURE_COLS].notna().all(axis=1)
    X_w = df.loc[not_derating, FEATURE_COLS][clean]
    y_w = y_look_1d[not_derating][clean]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_w, y_w, test_size=0.2, stratify=y_w, random_state=42
    )
    model_1d = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model_1d.fit(X_tr, y_tr)
    probs_1d = model_1d.predict_proba(X_te)[:, 1]

    precision_vals, recall_vals, _ = precision_recall_curve(y_te, probs_1d)
    fpr, tpr, _                    = roc_curve(y_te, probs_1d)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(fpr, tpr, color='steelblue', lw=2.5,
                 label=f'1-Day Model  (AUC={auc_1day:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random guess')
    axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
    axes[0].set_xlabel('False Positive Rate (safe trucks wrongly flagged)')
    axes[0].set_ylabel('True Positive Rate (real derates caught)')
    axes[0].set_title('ROC Curve -- 1 Day Before Derate')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(recall_vals, precision_vals, color='#DD4444', lw=2.5)
    axes[1].set_xlabel('Recall  (fraction of real derates caught)')
    axes[1].set_ylabel('Precision  (fraction of alerts that are real derates)')
    axes[1].set_title('Precision-Recall Trade-off -- 1 Day Before Derate')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print('Moving right on the PR curve = catch more derates but send more false alarms.')
    print('Moving up on the PR curve    = fewer false alarms but miss more derates.')
    print('Pick a threshold based on the operational cost of each type of error.')